In [1]:
import torch
import torch.nn as nn
import pytorch_lightning as pl
from transformers import BertModel, BertTokenizer
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

2025-06-21 07:08:23.960641: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750489704.153154      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750489704.209969      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
quora=pd.read_csv("/kaggle/input/quora-dataset/quora.csv",sep=",")
quora

,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0
...,...,...,...,...,...,...
404285,404285,433578,379845,How many keywords are there in the Racket prog...,How many keywords are there in PERL Programmin...,0
404286,404286,18840,155606,Do you believe there is life after death?,Is it true that there is life after death?,1
404287,404287,537928,537929,What is one coin?,What's this coin?,0
404288,404288,537930,537931,What is the approx annual cost of living while...,I am having little hairfall problem but I want...,0


In [3]:
import re
def clean_text(s):
    try:
        return re.sub(r'[^A-Za-z0-9,?"\'. ]+', '', s).encode('utf-8').decode('utf-8').lower()
    except:
        return ""
q1_processed=quora["question1"].apply(lambda x:clean_text(x))
q2_processed=quora["question2"].apply(lambda x:clean_text(x))

In [4]:
quora_processed=pd.concat([q1_processed,q2_processed],axis=1).reset_index(drop=True)
quora_processed

,question1,question2
0,what is the step by step guide to invest in sh...,what is the step by step guide to invest in sh...
1,what is the story of kohinoor kohinoor diamond?,what would happen if the indian government sto...
2,how can i increase the speed of my internet co...,how can internet speed be increased by hacking...
3,why am i mentally very lonely? how can i solve...,find the remainder when math2324math is divide...
4,"which one dissolve in water quikly sugar, salt...",which fish would survive in salt water?
...,...,...
404285,how many keywords are there in the racket prog...,how many keywords are there in perl programmin...
404286,do you believe there is life after death?,is it true that there is life after death?
404287,what is one coin?,what's this coin?
404288,what is the approx annual cost of living while...,i am having little hairfall problem but i want...


In [5]:
y = quora['is_duplicate']

In [6]:
x_train, x_test, y_train, y_test = train_test_split(quora_processed, y, test_size=0.2, random_state=42,stratify=y)

In [7]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [8]:
x_train=x_train.reset_index(drop=True)
x_test=x_test.reset_index(drop=True)
y_train=y_train.reset_index(drop=True)
y_test=y_test.reset_index(drop=True)

In [9]:
print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)
print(y.shape)

(323432, 2) (80858, 2) (323432,) (80858,)
(404290,)


In [10]:
class SiameseBert(pl.LightningModule):
    def __init__(self, model_name="bert-base-uncased", num_labels=2, learning_rate=1e-3,weight_decay=1e-4, similarity_metric="classifier"):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name)
        self.tokenizer = BertTokenizer.from_pretrained(model_name)
        self.learning_rate = learning_rate
        self.similarity_metric = similarity_metric
        self.weight_decay=weight_decay
        for param in self.bert.parameters():
            param.requires_grad=False
        
        
        if similarity_metric == "classifier":
            self.classifier = nn.Linear(self.bert.config.hidden_size * 3, num_labels) 
        
        self.loss_fn = nn.CrossEntropyLoss() if similarity_metric == "classifier" else nn.BCEWithLogitsLoss()
        self.save_hyperparameters()

    def forward(self, input_ids_1, attention_mask_1, input_ids_2, attention_mask_2):
        # Encode both sentences
        outputs_1 = self.bert(input_ids_1, attention_mask=attention_mask_1)
        outputs_2 = self.bert(input_ids_2, attention_mask=attention_mask_2)
        
        
        emb_1 = outputs_1.last_hidden_state[:, 0, :]
        emb_2 = outputs_2.last_hidden_state[:, 0, :]
        
        if self.similarity_metric == "cosine":
            return torch.cosine_similarity(emb_1, emb_2, dim=-1)
        else:
            
            diff = torch.abs(emb_1 - emb_2)
            combined = torch.cat([emb_1, emb_2, diff], dim=-1) 
            return self.classifier(combined)

    def training_step(self, batch, batch_idx):
        input_ids_1, attention_mask_1, input_ids_2, attention_mask_2, labels = batch
        logits = self(input_ids_1, attention_mask_1, input_ids_2, attention_mask_2)
        
        if self.similarity_metric == "cosine":
            loss = self.loss_fn(logits, labels.float())
        else:
            loss = self.loss_fn(logits, labels) 
        
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        input_ids_1, attention_mask_1, input_ids_2, attention_mask_2, labels = batch
        logits = self(input_ids_1, attention_mask_1, input_ids_2, attention_mask_2)
        if self.similarity_metric == "cosine":
            preds = (logits > 0.5).long()
            loss = self.loss_fn(logits, labels.float())
        else:
            preds = torch.argmax(logits, dim=1)
            loss = self.loss_fn(logits, labels)
        
        acc = accuracy_score(labels.cpu(), preds.cpu())
        f1 = f1_score(labels.cpu(), preds.cpu(), average="macro")
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)
        self.log("val_f1", f1, prog_bar=True)
        return {"val_loss": loss, "val_acc": acc, "val_f1": f1}

    def configure_optimizers(self):
        return AdamW(self.parameters(), lr=self.learning_rate,weight_decay=self.weight_decay)

    def predict_step(self, batch, batch_idx):
        input_ids_1, attention_mask_1, input_ids_2, attention_mask_2 = batch
        return self(input_ids_1, attention_mask_1, input_ids_2, attention_mask_2)

In [11]:
from torch.utils.data import Dataset, DataLoader

class SiameseBertDataset(Dataset):
    def __init__(self, texts_1, texts_2, labels, tokenizer, max_length=128):
        self.texts_1 = texts_1
        self.texts_2 = texts_2
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts_1)

    def __getitem__(self, idx):
        text1 = str(self.texts_1[idx])
        text2 = str(self.texts_2[idx])
        label = int(self.labels[idx])

    # Tokenize and return tensors directly (not dicts)
        encoding1 = self.tokenizer(
        text1, 
        max_length=self.max_length, 
        padding="max_length", 
        truncation=True, 
        return_tensors="pt"
    )
        encoding2 = self.tokenizer(
        text2, 
        max_length=self.max_length, 
        padding="max_length", 
        truncation=True, 
        return_tensors="pt"
    )

        return (
        encoding1["input_ids"].squeeze(0),  # Remove batch dim
        encoding1["attention_mask"].squeeze(0),
        encoding2["input_ids"].squeeze(0),
        encoding2["attention_mask"].squeeze(0),
        torch.tensor(label, dtype=torch.long)
    )

In [12]:
train_dataset = SiameseBertDataset(x_train["question1"], x_train["question2"], y_train, tokenizer)
val_dataset = SiameseBertDataset(x_test["question1"], x_test["question2"], y_test, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,num_workers=3)
val_loader = DataLoader(val_dataset, batch_size=32,num_workers=3)

In [13]:
from pytorch_lightning import Trainer
model = SiameseBert(similarity_metric="classifier")
trainer = Trainer(
    max_epochs=2,
    accelerator="gpu",
    devices=-1,
    logger=True,
    precision="16-mixed"
)
trainer.fit(model, train_loader, val_loader)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

In [14]:
for i in range (11,5,-1):
    for param in model.bert.encoder.layer[i].parameters():
        param.requires_grad=True

In [15]:
trainer = Trainer(
    max_epochs=3,
    accelerator="gpu",
    devices=-1,
    logger=True,
    precision="16-mixed"
)
model.learning_rate=1e-5
model.weight_decay=1e-3
trainer.fit(model, train_loader, val_loader)

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]